# Run a trained policy in the MuJoCo simulator

This notebook runs a trained **ACT** policy on a simulated SO-101 arm with **Physical AI Runtime**. It shows the same steps as for a real robot, and it ends in a live dashboard where you
start the policy with one click and watch it work.

The arm sorts items in a **conveyor cell**: items ride a belt past it, and it has to drop each one into the right bin. **Red, blue and green items go into the bin of their colour, purple or cracked items into the reject bin.**

| Step | Physical AI Runtime |
|---|---|
| 1 | Start the simulation: `physicalai-mujoco start` runs as its own process and serves the robot, its cameras and a 3D viewer |
| 2 | Load the policy: `InferenceModel.from_pretrained` |
| 3 | Connect: `RobotRuntime` with the robot, the cameras and the policy as action source |
| 4 | Run: `runtime.run()` / `runtime.stop()`, from a dashboard |

The policy was trained with [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) on about 100 episodes recorded from the simulation's built-in demonstrator, then exported to OpenVINO.

## Setup

Until the packages are published, run this notebook from a checkout of the repository, with its own environment:

```bash
uv sync --all-packages --extra notebooks
uv run jupyter lab packages/physicalai-mujoco-plugin/examples
```

(Once `physicalai` and the plugin are on PyPI, `pip install "physicalai[plugin-mujoco,transport,notebooks]"` is all you need. `conveyor_dashboard.py` next to this notebook holds the widgets, so the notebook only shows the Runtime code.)

In [ ]:
import os
import subprocess
import sys
import tempfile

from conveyor_dashboard import ConveyorDashboard, read_ready

# ACT policy, OpenVINO export. A Hub id, or a local folder with `manifest.json` via ACT_MODEL_DIR.
MODEL_REPO = os.environ.get("ACT_MODEL_DIR") or "iMaxxgx/act-so101-conveyor-sort"

## 1. Start the simulation

`physicalai-mujoco start` loads the scene and serves the robot to other processes by name (`--name`), its cameras and the 3D viewer. `--status-json` makes it report progress as JSON lines, ending with a `ready` event that
lists the ports it picked (`--http-port 0` and `--viser-port 0` mean "any free port"). `--exit-with-parent` stops it when this notebook's pipe to it closes, so it cannot be left running by accident.

In [ ]:
log = tempfile.NamedTemporaryFile("w+", prefix="mujoco-sim-", suffix=".log", delete=False)  # the simulation's own log
sim = subprocess.Popen(
    [
        sys.executable, "-m", "physicalai_mujoco_plugin", "start",
        "--profile", "so101", "--scene", "conveyor_sort", "--name", "notebook-sim",
        "--status-json", "--exit-with-parent", "--http-port", "0", "--viser-port", "0",
    ],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=log, text=True,
)
ready = read_ready(sim, log.name)
ready

## 2. Load the policy

`InferenceModel.from_pretrained` downloads the policy package from the Hugging Face Hub (an OpenVINO model plus a `manifest.json` that says which inputs it needs and how to preprocess them) and compiles it.

In [ ]:
from physicalai.inference import InferenceModel

policy = InferenceModel.from_pretrained(MODEL_REPO, backend="openvino", device="CPU")

## 3. Connect the robot, the cameras and the policy

A `RobotRuntime` ties them together, as on hardware:

- the **robot**: `SharedRobot` attaches to the simulation by its name,
- the **cameras**: the simulation's MJPEG streams, served like a real robot's IP cameras. Their names (`overview`, `gripper`) must match the policy's inputs,
- the **action source**: `PolicySource` runs the policy. ACT predicts a chunk of 100 actions, and `request_threshold=0.98` asks for a fresh chunk after about every second action, which this policy needs (it does not recover from a missed grasp by itself),
- the **loop rate**: 30 Hz.

In [ ]:
from physicalai.capture import IPCamera
from physicalai.robot import SharedRobot
from physicalai.runtime import ChunkedActionQueue, LerpSmoother, PolicySource, RobotRuntime, SyncExecution

http = ready["http_url"]
runtime = RobotRuntime(
    robot=SharedRobot(name=ready["name"]),
    cameras={
        "overview": IPCamera(url=f"{http}/cameras/overview/mjpeg", width=640, height=480, fps=30),
        "gripper": IPCamera(url=f"{http}/cameras/wrist/mjpeg", width=640, height=480, fps=30),
    },
    action_source=PolicySource(
        model=policy,
        execution=SyncExecution(request_threshold=0.98),
        action_queue=ChunkedActionQueue(smoother=LerpSmoother()),
    ),
    fps=30.0,
)
runtime.connect()  # attaches to the simulation and the cameras

## 4. Run it

The dashboard shows the simulation's 3D viewer (drag to orbit, scroll to zoom), the two cameras the policy sees and the score. The score counts items that ended up in the bin they belong in (**correct**), in a wrong bin (**wrong**), or off the belt (**missed**).

- **Run policy** runs `runtime.run()` and **Stop** calls `runtime.stop()`. The belt moves only while the policy runs, so it never starts with a full belt.
- **Reset** clears the belt and starts a new episode. **Belt cm/s** sets how fast items arrive: the policy was trained at 3 cm/s, so expect more misses when you speed it up.
- **Shut down** stops the policy and the simulation.

> Running Jupyter on another machine? The viewer URL points at *that* machine's `localhost`. Forward the port, or start the simulation with `--viser-host 0.0.0.0`.

In [ ]:
def shutdown():
    runtime.stop()
    runtime.disconnect()
    sim.stdin.close()  # closing the pipe stops the simulation (--exit-with-parent)
    sim.wait(timeout=30)


dashboard = ConveyorDashboard(
    http, ready["viewer_url"], run=runtime.run, stop=runtime.stop, shutdown=shutdown
)
dashboard.reset()  # the belt starts moving as soon as the simulation starts: clear it and hold it until Run
dashboard

## Where next

- Start another robot or scene: `physicalai-mujoco profiles` lists the robots, and the [plugin README](../README.md) describes the scenes.
- Record your own demonstrations, train in Physical AI Studio and point `MODEL_REPO` at your own export.
- Run the same policy from the command line: `physicalai run --config runtime.yaml`.